# Platform LLM operations

Application teams consume the lifecycle; the platform team operates the
substrate it runs on. This lab is the operator's loop for managing LLMs on
the platform:

1. Govern the provider catalog and the approved judge model.
2. Attribute every gateway request to an application and cost center.
3. Query spend by tag in the billing system tables.
4. Watch the fleet through provenance stamps and application manifests.
5. Oversee monitoring adoption and curate reviewed feedback into governed
   regression datasets.
6. Run the connected operator checks, and know the rollback levers.

The ownership split is explicit: application teams own evaluation cases,
scorer intent, and thresholds; the platform team owns judge deployments,
scorer configuration controls, sampling guardrails, dashboards, and alert
routing. Doctrine lives in [platform operations](../docs/platform-operations.md)
and the [LLMOps playbook](../docs/llmops-playbook.md); the deterministic cells
here run with zero credentials.

## 1. Govern the provider catalog

Applications name logical resources; the platform team decides what those
names resolve to. A project's `aai-platform.yml` carries only logical names
and governed deployments:

```yaml
providers:
  models:
    general-chat:
      provider: databricks
      deployment: approved-chat-endpoint
    judge-model:
      provider: databricks
      deployment: approved-judge-endpoint
```

Judges are measurements, so they are governed hardest: every LLM scorer must
resolve through the approved `judge-model`, never a provider's ambient
default. A Foundry model becomes a judge only behind a governed Databricks
external-model endpoint, which keeps authentication, gateway policy, and cost
controls platform-owned. Supported dependency ranges for the whole fleet are
pinned in `dependency-policy.toml`.

In [ ]:
from types import SimpleNamespace

from aai_core.evaluation import judge_model_uri
from aai_core.providers.types import ProviderConfigurationError

approved = SimpleNamespace(
    models={
        "judge-model": {
            "provider": "databricks",
            "deployment": "approved-judge-endpoint",
        }
    }
)
print("judge resolves to:", judge_model_uri(approved))

ungoverned = SimpleNamespace(
    models={"judge-model": {"provider": "foundry", "deployment": "raw-vendor"}}
)
try:
    judge_model_uri(ungoverned)
except ProviderConfigurationError as error:
    print(f"[{error.code}] {error}")

## 2. Attribute every gateway request

Classic compute carries the nine cost tags as cluster tags, serverless
workloads carry usage policies, and individual model requests through the
Databricks AI Gateway carry a request-tag header projected from the same
`ResourceContext`. The header deliberately excludes end-user identity:
shared application traffic stays application-attributed, and personal
identifiers never become billing metadata.

In [ ]:
from aai_core.tags import databricks_ai_gateway_request_headers
from aai_core.testing import dev_settings

for name, value in databricks_ai_gateway_request_headers(
    dev_settings().resource
).items():
    print(f"{name}: {value}")

## 3. Query spend by tag

Cluster tags and gateway request tags land in
`system.billing.usage.custom_tags`, so cost questions become governed SQL
rather than spreadsheet archaeology:

```sql
SELECT
  usage_date,
  custom_tags['application'] AS application,
  custom_tags['cost_center'] AS cost_center,
  SUM(usage_quantity) AS usage_quantity
FROM system.billing.usage
WHERE custom_tags['team'] IS NOT NULL
GROUP BY usage_date, application, cost_center
ORDER BY usage_date DESC
```

Unknown cost is never zero cost: evaluation records cost coverage
explicitly, and a release gate either requires a declared minimum coverage
or records that cost is report-only. The tag vocabulary is the
[tagging standard](../docs/tagging-standard.md).

## 4. Watch the fleet through provenance and manifests

Every generated project carries two machine-readable fleet hooks: the
`.aai-template.json` provenance stamp (which template and SDK version
produced it) and the `ai-app.yaml` manifest (owner, risk tier, evaluation
profile, readiness profile). The platform console renders both as the fleet
view; the stamps below are what a governed inventory reads.

In [ ]:
import json

provenance_stamp = {
    "template": "prompt-app",
    "template_version": "1.1.0",
    "supersedes": [],
    "generated_with": {
        "project": "fictional-earnings",
        "application": "earnings-summary",
        "team": "fictional-app-team",
        "providers": ["databricks"],
        "sdk": "0.3.0",
    },
}
required = {"template", "template_version", "generated_with"}
missing = sorted(required - set(provenance_stamp))
print("provenance complete" if not missing else f"missing keys: {missing}")
print(json.dumps(provenance_stamp, indent=2, sort_keys=True))

## 5. Oversee monitoring adoption and curate feedback

Production quality means the same thing as CI quality only when the same
scorer definitions run in both places — that is why the deterministic
scorers live in `aai_core.scorers`. Sampled-scorer registration
(`Scorer.register()` / `.start()`) runs from a Databricks notebook because
the service serializes notebook code; the platform team sets sampling rates
and judge cost budgets there, resolving the judge with `judge_model_uri()`.
Register only `as_mlflow_scorers(MONITORING_SCORERS)` for sampled traces:
production requests carry no ground-truth expectations, so the
reference-based scorers (`keyword_coverage`, `refusal_compliance`) belong
in offline evaluation and expectation-bearing regression datasets, never
in bare trace monitoring.

Reviewed feedback closes the loop. Feedback carries explicit non-personal
provenance (`group:domain-reviewers`, never an email address), and traces
flagged by reviewers become regression records in the governed evaluation
dataset. The filter below is pure so it runs anywhere; the connected cell
shows the full curation path.

In [ ]:
from types import SimpleNamespace

from aai_core.monitoring import traces_with_feedback

reviewed = [
    SimpleNamespace(
        trace_id="trace-101",
        info=SimpleNamespace(
            assessments=[SimpleNamespace(name="correct", value=False)]
        ),
    ),
    SimpleNamespace(
        trace_id="trace-102",
        info=SimpleNamespace(
            assessments=[SimpleNamespace(name="correct", value=True)]
        ),
    ),
]
flagged = traces_with_feedback(reviewed, name="correct", value=False)
print("flagged for the regression dataset:", [t.trace_id for t in flagged])

## 6. Run the connected operator checks

Flip the flag after keyless workspace access is in place
(`05_connected_setup.ipynb`). The cell runs the doctor with cloud checks,
confirms the operator identity, and demonstrates the curation path from
flagged production traces into the governed regression dataset.

Rollback levers, in order of preference: move the prompt alias back to the
previously adopted version (with the evidence that adopted it), redeploy the
previous immutable application release, and — for identity incidents —
follow the revocation runbook in [cloud setup](../docs/cloud-setup.md).
Releases are immutable, so rolling back never rewrites history.

In [ ]:
RUN_PLATFORM_CHECKS = False

if RUN_PLATFORM_CHECKS:
    from aai_core import bootstrap
    from aai_core.diagnostics import run_doctor
    from aai_core.evaluation import get_or_create_evaluation_dataset
    from aai_core.monitoring import traces_with_feedback

    for check in run_doctor(check_cloud=True):
        print(f"{check.status:>5}  {check.name}: {check.detail}")

    context = bootstrap()
    operator = context.workspace.current_user.me()
    print("operating as:", operator.user_name)

    mlflow = context.experiments.native_client
    experiment_name = context.settings.effective_experiment_name
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        print(
            f"no experiment exists yet at {experiment_name}; run a "
            "connected example (06 or 07) first so there are traces to "
            "curate"
        )
    else:
        traces = mlflow.search_traces(
            experiment_ids=[experiment.experiment_id], return_type="list"
        )
        flagged = traces_with_feedback(traces, name="correct", value=False)
        dataset = get_or_create_evaluation_dataset(
            name="production_regression_v1",
            catalog=context.settings.catalog,
            schema=context.settings.schema_name,
            experiment_id=experiment.experiment_id,
        )
        # The Databricks-managed dataset accepts record dictionaries, not
        # native Trace objects, so curate explicit inputs from each flagged
        # request and carry the trace's still-valid expectation assessments
        # along so the shared scorers keep testing the condition that
        # flagged it.
        import json

        records = []
        for trace in flagged:
            raw_request = trace.data.request
            try:
                decoded = json.loads(raw_request) if raw_request else {}
            except ValueError:
                decoded = raw_request
            inputs = (
                decoded if isinstance(decoded, dict) else {"request": decoded}
            )
            expectations = {}
            for assessment in trace.search_assessments(type="expectation"):
                value = getattr(
                    getattr(assessment, "expectation", None), "value", None
                )
                if value is not None:
                    expectations[assessment.name] = value
            record = {"inputs": inputs}
            if expectations:
                record["expectations"] = expectations
            records.append(record)
        if records:
            dataset.merge_records(records)
        print(
            f"curated {len(records)} reviewed traces into the governed "
            "dataset"
        )